# Task 2: Pop Song Melody Harmonization

## What is Harmonization?

Harmonization is the process of generating accompanying musical voices given a melody. In this task, we take a **melody track** from a pop song and generate a **piano accompaniment** that harmonizes it.

### Key Differences from Task 1 (Bach Chorales)

| Aspect | Task 1 (Bach) | Task 2 (Pop909) |
|--------|---------------|-----------------|
| **Genre** | Classical (1650-1750) | Modern Pop (various era) |
| **Model** | Transformer (causal, decoder-only) | Seq2Seq with Attention (encoder-decoder) |
| **Conditioning** | Unconditioned generation | Melody-conditioned (prefix encoding) |
| **Voices** | 4 voices (SATB) interleaved | Melody → Piano (variable voicing) |
| **Dataset size** | ~370 chorales | 909 pop songs, 70k+ training windows |
| **Harmony style** | Voice leading rules (parallel 5ths forbidden) | Modern pop (more flexible) |
| **Output** | Monophonic token sequence | 4 simultaneous pitches per timestep |

### Why Seq2Seq?

The seq2seq architecture is natural for harmonization because:
1. The **encoder** (bidirectional LSTM) reads the full melody to understand context
2. The **attention mechanism** focuses on relevant melody moments when generating piano notes
3. The **decoder** (unidirectional LSTM) generates 4 voices simultaneously, conditioned on the melody
4. Unlike the causal transformer, this allows the model to "look ahead" in the melody during encoding


## 2. Exploratory Data Analysis (EDA)

Let's load and visualize the POP909 dataset statistics.

In [ ]:

import json
import pandas as pd
from IPython.display import Image, display

# Load EDA results
with open('EDA_task2/pop909_eda_results.json', 'r') as f:
    eda_results = json.load(f)

# Display basic dataset info
dataset_info = eda_results['dataset_info']
print("=" * 60)
print("POP909 Dataset Overview")
print("=" * 60)
print(f"Training windows: {dataset_info['num_train_windows']:,}")
print(f"Validation windows: {dataset_info['num_val_windows']:,}")
print(f"Total windows: {dataset_info['num_train_windows'] + dataset_info['num_val_windows']:,}")
print(f"Unique chord types: {dataset_info['num_chords']}")
print(f"Melody pitch range: MIDI {dataset_info['melody_min_pitch']} – {dataset_info['melody_max_pitch']}")
print(f"Melody mean pitch: {dataset_info['melody_mean_pitch']:.1f}")
print()


### Melody Pitch Distribution

The melody track defines the harmonic content. We need to understand its pitch range and distribution to design appropriate decoding strategies.


In [ ]:

# Display melody pitch distribution
display(Image('images/task2_melody_pitch_distribution.png'))
print("Melody pitches span from C1 (MIDI 36) to G5 (MIDI 103)")
print(f"Mean melody pitch: {eda_results['melody_stats']['mean_pitch']:.1f}")
print(f"Median melody pitch: {eda_results['melody_stats']['median_pitch']:.1f}")


### Chord Vocabulary

Pop songs use a diverse set of chord types (major, minor, dominant 7th, suspended, etc.). This analysis shows how many unique chords appear in the POP909 dataset.


In [ ]:

display(Image('images/task2_chord_vocabulary.png'))
chord_stats = eda_results['chord_stats']
print(f"Unique chord types in training set: {chord_stats['num_unique_chords']}")
print(f"Top 5 most common chords:")
for i, (chord, count) in enumerate(chord_stats['top_chords'][:5], 1):
    print(f"  {i}. {chord}: {count:,} occurrences")


### Song Length Distribution

Song duration affects training dynamics. Longer songs provide more sequential data but may be harder to model.


In [ ]:

display(Image('images/task2_song_length.png'))
print(f"Shortest song: {eda_results['song_stats']['min_length']:.1f} seconds")
print(f"Longest song: {eda_results['song_stats']['max_length']:.1f} seconds")
print(f"Mean song length: {eda_results['song_stats']['mean_length']:.1f} seconds")


### Note Density

Note density (number of onsets per beat) varies across songs and affects harmony complexity.


In [ ]:

display(Image('images/task2_note_density.png'))
print(f"Mean note density: {eda_results['song_stats']['mean_note_density']:.2f} notes per beat")
print(f"Max note density: {eda_results['song_stats']['max_note_density']:.2f} notes per beat")


## 3. Modeling Architecture

### Seq2Seq with Attention

Our approach combines an **encoder** (reads the full melody) with a **decoder** (generates harmonies):

```
┌────────────────────────────────────────────┐
│         MELODY ENCODER (Bidirectional)     │
│  Input: Melody pitch sequence              │
│  [Embedding] → [BiLSTM × 2 layers]        │
│  Output: Context vectors + hidden states   │
└────────────────────────────────────────────┘
                     ↓
        ┌────────────────────────┐
        │  BAHDANAU ATTENTION    │
        │  (Focus on relevant    │
        │   melody moments)      │
        └────────────────────────┘
                     ↓
┌────────────────────────────────────────────┐
│         CHORD DECODER (Unidirectional)     │
│  Input: Previous chord + attention context │
│  [LSTM × 2 layers]                        │
│  Output: 4 pitch predictions (0-128)      │
│  ├─ Alto head (linear)                    │
│  ├─ Tenor head (linear)                   │
│  ├─ Bass head (linear)                    │
│  └─ Soprano (given from input)            │
└────────────────────────────────────────────┘
                     ↓
         Piano accompaniment (4 voices)
```

### Model Parameters


In [ ]:

import sys
sys.path.insert(0, 'modeling_task2')

from harmonizer_model import HarmonizerSeq2Seq
import torch

# Instantiate the model
model = HarmonizerSeq2Seq(
    melody_embed_dim=128,
    melody_hidden_dim=256,
    chord_embed_dim=128,
    chord_hidden_dim=256,
    num_layers=2,
    dropout=0.3
)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=" * 60)
print("Model Architecture: HarmonizerSeq2Seq")
print("=" * 60)
print(model)
print()
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print()
print("Component breakdown:")
print(f"  • MelodyEncoder: ~{sum(p.numel() for p in model.melody_encoder.parameters()):,} params")
print(f"  • BahdanauAttention: ~{sum(p.numel() for p in model.attention.parameters()):,} params")
print(f"  • ChordDecoder: ~{sum(p.numel() for p in model.chord_decoder.parameters()):,} params")


### Training Details

**Loss Function:** 4-voice Cross-Entropy
- Each timestep predicts 4 pitches (Alto, Tenor, Bass independently; Soprano is given)
- Loss = CE(Alto) + CE(Tenor) + CE(Bass) averaged over the sequence

**Teacher Forcing:** During training, we provide ground-truth previous chords as input.
During inference, we use the model's own predictions (autoregressive).

**Bidirectional Encoding:** The encoder is bidirectional (reads left-to-right and right-to-left),
allowing it to see the full melody context before generating harmonies. This is different from
the causal transformer in Task 1, which can only attend to previous tokens.

**Attention Mechanism:** Bahdanau-style additive attention allows the decoder to focus on
different parts of the melody at each generation step, enabling dynamic harmony matching.


In [ ]:

# Check if losses file exists
import os

losses_file = 'modeling_task2/harmonizer_losses.json'
if os.path.exists(losses_file):
    with open(losses_file, 'r') as f:
        losses = json.load(f)

    import matplotlib.pyplot as plt

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # Training vs Validation Loss
    ax1.plot(losses['train_loss'], label='Training Loss', linewidth=2)
    ax1.plot(losses['val_loss'], label='Validation Loss', linewidth=2)
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training Curves')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Final loss values
    ax2.bar(['Training', 'Validation'],
            [losses['train_loss'][-1], losses['val_loss'][-1]],
            color=['blue', 'orange'])
    ax2.set_ylabel('Loss')
    ax2.set_title('Final Loss Values')
    ax2.set_ylim([0, max(losses['val_loss'])])

    for i, v in enumerate([losses['train_loss'][-1], losses['val_loss'][-1]]):
        ax2.text(i, v + 0.02, f'{v:.4f}', ha='center', fontweight='bold')

    plt.tight_layout()
    plt.savefig('images/task2_training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f"Final training loss: {losses['train_loss'][-1]:.4f}")
    print(f"Final validation loss: {losses['val_loss'][-1]:.4f}")
else:
    print("⚠ Training losses not found.")
    print("Run the Colab training script first: colab/task2/train_harmonizer_colab.py")
    print()
    print("Expected output:")
    print("  • harmonizer_best.pt (checkpoint)")
    print("  • harmonizer_losses.json (training curves)")


## 4. Training Instructions (Google Colab)

Since the model requires GPU training, we provide a Colab notebook workflow.

### Step 1: Setup
```bash
# Clone repo and install dependencies
!git clone https://github.com/your-repo/153Assignment2.git
cd /content/153Assignment2
!pip install torch torchaudio librosa mido pretty_midi -q
```

### Step 2: Upload Data
Upload `pop909_cache.pkl` to Colab:
```python
from google.colab import files
files.upload()  # Select pop909_cache.pkl
```

### Step 3: Run Training
Copy cells from `colab/task2/train_harmonizer_colab.py` into your Colab notebook.

Key parameters:
- **Batch size:** 32 (adjust for your GPU)
- **Learning rate:** 0.001 (Adam optimizer)
- **Epochs:** 20 (or more if validation loss still decreasing)
- **GPU:** T4 (Google Colab free tier) or A100 (preferred, ~15 min/epoch)

### Step 4: Download Checkpoint
```python
from google.colab import files
files.download('modeling_task2/harmonizer_best.pt')
```

Upload the downloaded checkpoint back to your local repo.

### Expected Training Time
- **T4 GPU:** ~60-90 minutes for 20 epochs
- **A100 GPU:** ~15-20 minutes for 20 epochs

### Monitoring
The Colab script will save:
- `harmonizer_best.pt` (best validation checkpoint)
- `harmonizer_losses.json` (training curves)
- `best_config.json` (hyperparameters used)


## 5. Evaluation & Generated Samples

### Inference Strategy

We use constrained decoding to ensure generated harmonies stay in the same key as the melody:
1. Load melody track from test song
2. Run encoder over full melody sequence
3. At each timestep, decoder uses attention + previous chord to predict next chord
4. Post-filter: Keep only pitches in the detected key

### Generated Harmonies

Here are examples of our model harmonizing pop melodies:


In [ ]:

import os
from pathlib import Path

# List available MIDI files
eval_dir = Path('evaluation_task2')
midi_files = sorted(eval_dir.glob('harmony_*.mid'))

if midi_files:
    print(f"Generated {len(midi_files)} harmonizations:")
    for mf in midi_files:
        print(f"  • {mf.name}")
else:
    print("No generated MIDI files found yet.")
    print("Run inference first: python modeling_task2/generate_harmony.py")


### Audio Examples

Load generated MIDI and convert to WAV for listening:


In [ ]:

import subprocess
from pathlib import Path
import base64

def midi_to_wav(midi_path, wav_path, soundfont='modeling/checkpoints/MuseScore_General.sf3'):
    """Convert MIDI to WAV using fluidsynth."""
    try:
        result = subprocess.run(
            ['fluidsynth', '-ni', soundfont, str(midi_path), '-F', str(wav_path), '-r', '44100'],
            capture_output=True,
            timeout=30
        )
        return result.returncode == 0
    except Exception as e:
        print(f"Error: {e}")
        return False

def embed_audio(wav_path, title="Audio"):
    """Create HTML audio player with base64-encoded WAV."""
    try:
        with open(wav_path, 'rb') as f:
            wav_data = f.read()
        b64 = base64.b64encode(wav_data).decode()

        html = f'''
        <div style="margin: 10px 0; padding: 10px; background: #f0f0f0; border-radius: 5px;">
            <p><strong>{title}</strong></p>
            <audio controls style="width: 100%;">
                <source src="data:audio/wav;base64,{b64}" type="audio/wav">
                Your browser does not support the audio element.
            </audio>
        </div>
        '''
        from IPython.display import HTML
        return HTML(html)
    except Exception as e:
        print(f"Error loading audio: {e}")
        return None

# Convert and embed examples
examples = [
    ('harmony_001.mid', 'Song 001 - Generated Harmony'),
    ('harmony_042.mid', 'Song 042 - Generated Harmony'),
    ('harmony_100.mid', 'Song 100 - Generated Harmony'),
]

soundfont_path = 'modeling/checkpoints/MuseScore_General.sf3'
if not os.path.exists(soundfont_path):
    print(f"⚠ Soundfont not found at {soundfont_path}")
    print("Download from: https://github.com/musescore/MuseScore/releases")
else:
    from IPython.display import HTML, display

    for midi_name, title in examples:
        midi_path = eval_dir / midi_name
        if midi_path.exists():
            wav_path = midi_path.with_suffix('.wav')

            # Convert if WAV doesn't exist
            if not wav_path.exists():
                print(f"Converting {midi_name} to WAV...")
                success = midi_to_wav(str(midi_path), str(wav_path), soundfont_path)
                if not success:
                    print(f"Failed to convert {midi_name}")
                    continue

            # Embed audio
            display(embed_audio(str(wav_path), title))
        else:
            print(f"MIDI file not found: {midi_path}")


## 6. Metrics & Baseline Comparison

We evaluate harmonization quality using multiple metrics:

### Metrics Explained

| Metric | Definition | Interpretation |
|--------|-----------|-----------------|
| **Scale Consistency** | % of notes in the predicted key | Higher is better; ensures harmonic coherence |
| **Parallel 5ths Rate** | % of voice pairs with parallel perfect 5ths | Lower is better; forbidden in classical voice leading |
| **Voice Crossing Rate** | % of timesteps with voice ordering violations | Lower is better; standard rule in chorale harmonization |
| **Pitch KL Divergence** | KL divergence of pitch class distribution vs. real Bach | Lower is better; measures statistical similarity |

### Results


In [ ]:

import json
import pandas as pd

metrics_file = 'evaluation_task2/harmony_metrics.json'

if os.path.exists(metrics_file):
    with open(metrics_file, 'r') as f:
        metrics = json.load(f)

    # Create comparison table
    df = pd.DataFrame({
        'Our Model': metrics.get('our_model', {}),
        'Random Baseline': metrics.get('random_baseline', {}),
        'Real Pop (Ground Truth)': metrics.get('real_bach', {})  # Reference from original
    }).T

    print("=" * 80)
    print("Harmonization Metrics Comparison")
    print("=" * 80)
    print(df.round(2))
    print()

    # Interpretation
    our = metrics.get('our_model', {})
    baseline = metrics.get('random_baseline', {})

    print("Interpretation:")
    print(f"✓ Scale Consistency: {our.get('scale_consistency', 0):.1f}% vs {baseline.get('scale_consistency', 0):.1f}% baseline")
    print(f"✓ Parallel 5ths: {our.get('parallel_5ths_rate', 0):.2f}% vs {baseline.get('parallel_5ths_rate', 0):.2f}% baseline")
    print(f"✓ Voice Crossing: {our.get('voice_crossing_rate', 0):.2f}% vs {baseline.get('voice_crossing_rate', 0):.2f}% baseline")
    print(f"✓ Pitch KL Divergence: {our.get('pitch_kl_divergence', 0):.3f} vs {baseline.get('pitch_kl_divergence', 0):.3f} baseline")
else:
    print("⚠ Metrics file not found at", metrics_file)
    print("Run evaluation after training: python evaluation_task2/evaluate_harmony.py")
    print()
    print("Placeholder metrics:")
    df_placeholder = pd.DataFrame({
        'Our Model': {
            'Scale Consistency': '—',
            'Parallel 5ths Rate': '—',
            'Voice Crossing Rate': '—',
            'Pitch KL Divergence': '—'
        },
        'Random Baseline': {
            'Scale Consistency': '69.9%',
            'Parallel 5ths Rate': '0.0%',
            'Voice Crossing Rate': '96.3%',
            'Pitch KL Divergence': '0.104'
        },
        'Real Pop': {
            'Scale Consistency': '92.7%',
            'Parallel 5ths Rate': '0.17%',
            'Voice Crossing Rate': '1.4%',
            'Pitch KL Divergence': '1.625'
        }
    }).T
    print(df_placeholder)


## 7. Related Work

### DeepBach (Hadjeres et al., 2017)
DeepBach pioneered neural harmonization for Bach chorales using **Gibbs sampling** and blocked conditional RBMs.
- **Approach:** Train separate models for each voice, then use Gibbs sampling to ensure voice coherence
- **Advantage:** Explicit constraint handling, interpretable voice-by-voice generation
- **Limitation:** Slow inference (requires many Gibbs iterations)
- **vs. Ours:** Our seq2seq is faster (single forward pass) but less principled in enforcing voice leading rules

### Coconet (Huang et al., 2017)
Coconet used **dilated CNNs** with **blocked Gibbs sampling** for polyphonic music generation.
- **Architecture:** Dilated convolutions to capture long-range dependencies efficiently
- **Advantage:** Can generate multiple voices in parallel, flexible blocking strategy
- **Limitation:** Still requires expensive sampling; harder to interpret attention patterns
- **vs. Ours:** Our attention mechanism is more interpretable; Coconet may capture longer dependencies

### Music Transformer (Huang et al., 2018)
Relative position-based attention for long-form unconditioned generation.
- **Approach:** Transformer with relative positional embeddings, trained on Maestro dataset
- **Advantage:** Can generate very long coherent sequences (up to 4 minutes)
- **Limitation:** No explicit melody conditioning; unconditioned generation
- **vs. Ours:** Task 1 (Bach) resembles this; Task 2 adds explicit melody prefix encoding

### Comparing to Pop Music Constraints
Pop songs have different harmonic constraints than Bach chorales:
- **Voice leading rules** are more relaxed (no strict prohibition on parallel 5ths)
- **Chord changes** are often synchronized with melody (homophonic texture)
- **Dynamics & timbre** matter more than in symbolic music (harder to capture in MIDI)

Our seq2seq approach works well for pop because:
1. Melody-conditioning matches the pop harmonic structure
2. Attention aligns harmonies with melody changes
3. No need for expensive Gibbs sampling → real-time inference possible
